In [ ]:
import gymnasium as gym
from gymnasium_env import BlokusEnv
from gymnasium_env.envs.single_agent_blokus_env import SingleAgentBlokusEnv
from gymnasium_env.envs.blokus_action import BlokusAction
from src.agents import Agent, QL_Agent, ABPruningAgent
import numpy as np
from tqdm import tqdm
import pickle
import cProfile
from src.agents.ab_pruning.heuristics import piece_size, level_7


In [ ]:
BOARD_SIZE = 7
PLAYER_TURN = 2

In [ ]:

env = SingleAgentBlokusEnv(
    board_size=BOARD_SIZE, 
    hidden_agents=[
        ABPruningAgent(name="hey",
                       board_size=BOARD_SIZE,
                       sorted_order=lambda x: level_7(*x), testing_mode=(True, 0)),
        # QL_Agent(name="hey", q_table_path="/Users/mario/Documents/proj/cam/Blokus/models/q_tables/7x7/q_table1.pkl"),
        None,
    ],
    player_turn=PLAYER_TURN,
    render_mode="human"
)

In [ ]:

# Create an instance of the custom environment
def test_agent(env, num_episodes=1):
    global q_table
    win_counter, tie_counter, lose_counter = 0, 0, 0
    for _ in range(num_episodes):
        obs, info = env.reset()
        # state = encode_board(obs["state"])
        done = False
        total_reward = obs["points"][PLAYER_TURN] - obs["points"][3 - PLAYER_TURN]
        while not done:
            with open("possible_actions.txt", "w") as file:
                file.write("----------------------------------------------\n")
                file.write("Possible actions:\n")
                actions = [BlokusAction(board_size=BOARD_SIZE, action_id=action_id) for action_id in obs['possible_actions']]
                for idx, action in enumerate(actions):
                    file.write(f"{idx} : {action}\n")
            action = actions[int(input("Enter action: "))]
            
            next_obs, reward, terminated, truncated, info = env.step(action.action_id)
            # next_state = encode_board(next_obs["state"])
            # state = next_state
            obs = next_obs
            total_reward += reward
            done = terminated or truncated
            # env.render()
        if total_reward > 0:
            win_counter += 1
        elif total_reward == 0:
            tie_counter += 1
        else:
            lose_counter += 1
        print(f"Player 1 (Machine) score: {obs['points'][1]}, Player 2 (You) score: {obs['points'][2]}")
    return win_counter, tie_counter, lose_counter

# win_counter1 = np.zeros(1000)
# tie_counter1 = np.zeros(1000)
# lose_counter1 = np.zeros(1000)
# # Test the agent
# for i in tqdm(range(1000)):
#     win_counter1[i], tie_counter1[i], lose_counter1[i] = test_agent(env)


# print(f"Number of not visited states: {not_visited}")

In [ ]:
test_agent(env, 1)

In [ ]:
# env.hidden_agents[1].cache_manager.save_cache()

test_agent(env, 1)